In [12]:
import sys, os, time, json, itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import xgboost as xgb

ROOT = os.path.abspath("..")
sys.path.append(ROOT)
DATA = os.path.join(ROOT, "data", "processed", "loans_clean.parquet")
FIG = os.path.join(ROOT, "reports", "figures")

from src.data import load_splits, for_trees
from src.metrics import evaluate

tr, va, te = load_splits(DATA)
((Xtr, ytr), (Xva, yva), (Xte, yte)), feats = for_trees(tr, va, te)
print(Xtr.shape, Xva.shape, Xte.shape, "| features:", len(feats))

(60298, 68) (12921, 68) (12922, 68) | features: 68


In [ ]:
def make(depth=6, lr=0.05, mcw=1):
    return xgb.XGBClassifier(
        n_estimators=2000, learning_rate=lr, max_depth=depth, min_child_weight=mcw,
        subsample=0.8, colsample_bytree=0.8, tree_method="hist", enable_categorical=True,
        early_stopping_rounds=50, eval_metric="auc", random_state=42, n_jobs=-1)

base = make()
base.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
print("best iteration:", base.best_iteration)
print("val:", evaluate(yva, base.predict_proba(Xva)[:, 1]))

In [ ]:
rows = []
for depth, lr, mcw in itertools.product([3, 5, 7], [0.03, 0.1], [1, 10]):
    m = make(depth, lr, mcw)
    t0 = time.time()
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    rows.append(dict(depth=depth, lr=lr, mcw=mcw, iters=m.best_iteration,
                     val_auc=evaluate(yva, m.predict_proba(Xva)[:, 1])["roc_auc"],
                     secs=round(time.time() - t0, 1)))
res = pd.DataFrame(rows).sort_values("val_auc", ascending=False)
res

In [ ]:
best = res.iloc[0]
final = make(int(best.depth), float(best.lr), int(best.mcw))
t0 = time.time()
final.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
train_secs = round(time.time() - t0, 1)

result = {
    "framework": "XGBoost",
    "params": {"max_depth": int(best.depth), "learning_rate": float(best.lr), "min_child_weight": int(best.mcw)},
    "train_seconds": train_secs,
    "val": evaluate(yva, final.predict_proba(Xva)[:, 1]),
    "test": evaluate(yte, final.predict_proba(Xte)[:, 1]),
}
print(json.dumps(result, indent=2))
with open(os.path.join(ROOT, "reports", "results_xgboost.json"), "w") as f:
    json.dump(result, f, indent=2)

In [ ]:
import shap
sample = Xte.sample(2000, random_state=42)
sv = shap.TreeExplainer(final).shap_values(sample)
shap.summary_plot(sv, sample, show=False)
plt.savefig(os.path.join(FIG, "xgb_shap_summary.png"), dpi=150, bbox_inches="tight")
plt.show()